# 02 · Event construction

From the price panel to 14,678 crash events, and the checks that the construction is causal.

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from deadcat import data as D, plotting as P
from deadcat.config import load_config
P.use_style(); pd.set_option("display.width", 180)
cfg = load_config(ROOT / "configs" / "default.yaml")
T = ROOT / "results" / "tables"; M = ROOT / "results" / "metrics"
print("config fingerprint:", cfg.fingerprint)

config fingerprint: a9ca33aed3d9


In [2]:
ev = D.load_processed("events.parquet")
ev["event_date"] = pd.to_datetime(ev["event_date"])
print(json.dumps({k: v for k, v in json.load(open(M / "event_construction.json")).items()
                  if k != "feature_missing_pct"}, indent=2))

{
  "config_fingerprint": "a9ca33aed3d9",
  "n_events": 14678,
  "n_events_complete_car20": 14647,
  "n_tickers": 600,
  "date_min": "2007-01-03",
  "date_max": "2026-08-28",
  "crash_type_counts": {
    "idiosyncratic": 8192,
    "broad_market": 4948,
    "sector": 1449,
    "unclassified": 89
  },
  "sector_known_pct": 99.1,
  "mean_crash_z": -4.389524731953498,
  "mean_raw_return": -0.06484600757916571
}


## The causality check that matters

μ and σ are estimated on the 60 days *strictly before* the event. The strong test: corrupt every return from *t* onward and confirm the statistics at *t* do not move.

In [3]:
from deadcat import events as E
px = D.load_processed("prices.parquet")
close = px.pivot(index="date", columns="ticker", values="close").sort_index()
ret = E.daily_returns(close[["AAPL"]])
t = 3000
mutated = ret.copy(); mutated.iloc[t:] = mutated.iloc[t:] * 100 + 7
mu_a, sd_a = E.rolling_moments(ret, 60, 60)
mu_b, sd_b = E.rolling_moments(mutated, 60, 60)
print(f"mu  at t: {mu_a.iloc[t, 0]:.8f} vs {mu_b.iloc[t, 0]:.8f}")
print(f"sigma at t: {sd_a.iloc[t, 0]:.8f} vs {sd_b.iloc[t, 0]:.8f}")
print("identical ->", np.isclose(mu_a.iloc[t, 0], mu_b.iloc[t, 0]) and
                      np.isclose(sd_a.iloc[t, 0], sd_b.iloc[t, 0]))

mu  at t: 0.00228845 vs 0.00228845
sigma at t: 0.00695349 vs 0.00695349
identical -> True


## Cooldown: events are independent by construction

In [4]:
cal = close.index
pos = pd.Series(np.arange(len(cal)), index=cal)
g = ev.assign(p=ev.event_date.map(pos)).sort_values(["ticker", "p"]).groupby("ticker")["p"].diff().dropna()
print(f"minimum trading-day gap between same-ticker events: {g.min():.0f} "
      f"(cooldown = {cfg.events.cooldown_days})")
print("violations:", int((g < cfg.events.cooldown_days).sum()))
print("duplicate event_id:", int(ev.event_id.duplicated().sum()))

minimum trading-day gap between same-ticker events: 20 (cooldown = 20)
violations: 0
duplicate event_id: 0


## Severity and timing

In [5]:
print(ev[["crash_z", "raw_return", "abs_decline", "hl_range", "avol"]].describe().T.round(4))
q = ev.set_index("event_date").resample("QE").size()
print("\nbusiest quarters:"); print(q.sort_values(ascending=False).head(5).to_string())

               count    mean     std      min     25%     50%     75%     max
crash_z      14678.0 -4.3895  1.9546 -45.8010 -4.7148 -3.7540 -3.2898 -3.0003
raw_return   14678.0 -0.0648  0.0417  -0.5904 -0.0764 -0.0536 -0.0394 -0.0140
abs_decline  14678.0  0.0648  0.0417   0.0140  0.0394  0.0536  0.0764  0.5904
hl_range     14678.0  0.0543  0.0328   0.0000  0.0349  0.0470  0.0645  0.5886
avol         14646.0  0.9215  0.5664  -1.0986  0.5209  0.8447  1.2453  4.5643

busiest quarters:
event_date
2020-03-31    629
2018-12-31    463
2025-06-30    386
2011-09-30    370
2018-03-31    368


## Crash-type classification

Operational, not causal.

In [6]:
display(ev.crash_type.value_counts().to_frame("events")
        .assign(share=lambda d: (d.events / len(ev) * 100).round(1)))

,events,share
crash_type,,
idiosyncratic,8192,55.8
broad_market,4948,33.7
sector,1449,9.9
unclassified,89,0.6


## Outcomes

In [7]:
c = ev[ev.complete_20]
cols = ["car_1", "car_5", "car_10", "car_20", "car_60", "mfe_20", "mae_20"]
display(c[cols].describe().T[["count", "mean", "50%", "std", "min", "max"]].round(4))
print(f"recovered_20d      : {c.recovered_20d.mean():.4f}")
print(f"regained pre-crash : {c.regained_precrash_20.mean():.4f}")
print(f"median days to recovery (of those that did): {c.days_to_recovery.median():.0f}")

,count,mean,50%,std,min,max
car_1,14647.0,-0.0010,-0.0008,0.0246,-0.4606,0.3277
car_5,14647.0,-0.0027,-0.0024,0.0466,-0.7868,0.6147
car_10,14646.0,-0.0019,-0.0019,0.0618,-0.6955,1.0645
car_20,14647.0,-0.0027,-0.0043,0.0840,-0.6254,2.3425
car_60,14539.0,-0.0043,-0.0069,0.1389,-0.6859,3.1649
mfe_20,14647.0,0.0570,0.0438,0.0750,-0.4787,2.9512
mae_20,14647.0,-0.0594,-0.0379,0.0838,-0.8884,0.2215


recovered_20d      : 0.4714
regained pre-crash : 0.3894
median days to recovery (of those that did): 9


**The dispersion is the finding.** σ(CAR₂₀) ≈ 8.4% around a −0.27% mean.